In [ ]:
# h3-pixels-crop-yield — shared config (%run ./config_nb). A job run strips the install cells and installs from its environment spec.
# GeoBrix: the env6 wheel from the v0.6.1 GitHub release (other installs: https://databrickslabs.github.io/geobrix/docs/installation).
%pip install --quiet "geobrix[vizx] @ https://github.com/databrickslabs/geobrix/releases/download/v0.6.1/geobrix-0.6.1+env6-py3-none-any.whl"
# No access to github.com: stage the wheel on a Volume and install it from there instead:
# %pip install --quiet "geobrix[vizx] @ file:///Volumes/<catalog>/<schema>/<volume>/geobrix-0.6.1+env6-py3-none-any.whl"
# earthaccess on its own line; then %restart_python (next cell).
%pip install --quiet "earthaccess>=0.11,<1" "botocore<1.44.0"

In [ ]:
%restart_python

In [ ]:
import functools
import os

from pyspark.databricks.sql import functions as DBF
from pyspark.sql import DataFrame
from pyspark.sql import functions as F

# GeoBrix: one Python API and one register call on every compute.
from databricks.labs import gbx
from databricks.labs.gbx.rasterx import functions as rx

_ = gbx.register(spark)  # the gbx_* SQL functions and the *_gbx readers

In [ ]:
# ============================================================
# USER SETTINGS — edit these (everything below is wired off them)
# ============================================================
catalog_name = "geospatial_docs"     # a Volume named 'data' must exist under catalog/schema
schema_name = "h3_pixels_crop_yield"

FULL_AOI = False                     # False -> Story + Boone counties; True -> all of Iowa (downloads + dataset layer)
SEASONS = [2023]                     # any of 2018-2025 (the CSB crop codes)
DEMO_COUNTIES = ("19169", "19015")   # Story, Boone
DEMO_FIELDS = 3500                   # corn field polys nearest the Story County centre (2023: 5,250 in the two counties)
FIELD_H3_RES = 13                    # field cell resolution
RES_RULE = "not_larger"              # gbx_rst_h3_fitresolution rule: 'not_larger' or 'log_nearest'
SETBACK_M = 10                       # border setback: keep field cells at least this far inside the edge
EDGE_WEIGHT_D_M = 30                 # D in the edge weight w = min(1, d / D)
HLS_MASK_BITS = (1,)                 # Fmask bits set to NoData (bit 1 = cloud)
USE_NCLIMGRID = False                # also download nClimGrid monthly precipitation as a cross-check
EARTHDATA_UC_SECRET = "geospatial_docs.vapor_eyes.earthdata_token"  # catalog.schema.key of an Earthdata Login token
FORCE_REBUILD = False                # True -> re-download and rebuild every table
INTERACTIVE_PLOTS = False            # False -> static maps; True -> MapLibre

# Area of interest (EPSG:4326): Story + Boone + 5 km, and all of Iowa, from the corners of the CSB polys' EPSG:5070
# bbox (so slightly larger than the true lon/lat bbox).
DEMO_BBOX = (-94.2434, 41.7965, -93.154, 42.2734)
IOWA_BBOX = (-96.646, 40.3287, -90.0372, 43.5833)
STORY_CENTROID = (-93.466093, 42.037538)  # the Census 2023 internal point of Story County (lon, lat)


def get_aoi_bbox():
    """The AOI bbox from the CURRENT FULL_AOI toggle, re-read on every call."""
    return IOWA_BBOX if FULL_AOI else DEMO_BBOX


def aoi_wkt():
    w, s, e, n = get_aoi_bbox()
    return f"POLYGON (({w} {s}, {e} {s}, {e} {n}, {w} {n}, {w} {s}))"


# Months scored per dataset (May-September; groundwater May-August; NDVI August).
MONTHS = {"stage4": (5, 6, 7, 8, 9), "normals": (5, 6, 7, 8, 9), "nclimgrid": (5, 6, 7, 8, 9),
          "gws_inst": (5, 6, 7, 8), "sfsm_inst": (5, 6, 7, 8, 9), "gridmet": (5, 6, 7, 8, 9), "hls": (8,)}
LINK_K = 1            # k-ring of candidate dataset cells for a field cell that straddles (k = 1 holds every one)
RETILE_PX = 256       # sub-tile size before gridding (per-task memory)
FANOUT = 512          # Serverless parallelism: repartition(FANOUT, column)
# Raster-to-H3 coverage per dataset: HLS keeps only cells with a valid (cloud-free) pixel; the rest keep every
# covered cell, with a NULL measure where no pixel is valid.
GRID_COVERAGE = {"hls_l30": "sparse", "hls_s30": "sparse"}  # default 'complete'
NDVI_ENGINE = "mapalgebra"  # 'mapalgebra' (session and UC) or 'derivedband' (session only)
WIND_SCALE = 1.0      # gridMET wind_speed: the reader returns m/s (the file's counts x 0.1)
NORMALS_TO_MM = 1.0   # the normals file is in millimetres

# Sources (public HTTPS, no login). A mirror must sit on the same host: each downloader has an HTTP allow-list.
STAGE4_URL_TEMPLATE = ("https://water.noaa.gov/resources/downloads/precip/stageIV/{y:04d}/{m:02d}/{d:02d}/"
                       "nws_precip_1day_{y:04d}{m:02d}{d:02d}_conus.nc")
NORMALS_URL = ("https://www.ncei.noaa.gov/data/oceans/archive/arc0196/0248761/1.1/data/0-data/"
               "prcp-2006_2020-monthly-normals-v1.0.nc")
NCLIMGRID_URL = "https://noaa-nclimgrid-monthly-pds.s3.amazonaws.com/nclimgrid_prcp.nc"
GRIDMET_BASE = "https://www.northwestknowledge.net/metdata/data"  # files vs_<YYYY>.nc
CSB_ZIP_URL = ("https://www.nass.usda.gov/sites/default/files/Research_and_Science/Crop-Sequence-Boundaries/datasets/"
               "NationalCSB_2018-2025_rev23.zip")

In [ ]:
# Earthdata token -> env (guarded; only HLS and GRACE-DA need it).
_cat, _sch, _name = EARTHDATA_UC_SECRET.split(".")
_tok = None
try:
    _tok = dbutils.secrets.get(_cat, _sch, _name)  # UC secret (3-arg)
except Exception:
    _tok = None
if _tok:
    os.environ["EARTHDATA_TOKEN"] = _tok
    print(f"... EARTHDATA_TOKEN loaded from {EARTHDATA_UC_SECRET}")
else:
    print(f"... EARTHDATA_TOKEN NOT set. HLS and GRACE-DA need UC secret {EARTHDATA_UC_SECRET}; the other datasets run without it.")
_tok = None  # clear the value from the notebook namespace (the env var keeps it)
EARTHDATA_READY = bool(os.environ.get("EARTHDATA_TOKEN"))

In [ ]:
# Spark-conf tuning, guarded for Serverless (a no-op there).
def set_conf_safe(key, value):
    try:
        spark.conf.set(key, value)
        return True
    except Exception as e:
        print(f"... skipping spark.conf.set({key}) [Serverless?]: {type(e).__name__}")
        return False


set_conf_safe("spark.sql.shuffle.partitions", 512)

In [ ]:
spark.sql(f"USE CATALOG {catalog_name}")
spark.sql(f"CREATE DATABASE IF NOT EXISTS {schema_name}")
spark.sql(f"USE DATABASE {schema_name}")

In [ ]:
# Volume ETL tree (Volume 'data' must exist).
ETL_DIR = f"/Volumes/{catalog_name}/{schema_name}/data"
H3PX_DIR = f"{ETL_DIR}/h3-pixels-crop-yield"
CSB_DIR, STAGE4_DIR, NORMALS_DIR = f"{H3PX_DIR}/csb", f"{H3PX_DIR}/stage4", f"{H3PX_DIR}/normals"
NCLIMGRID_DIR, GRACE_DIR, GRIDMET_DIR, HLS_DIR = (f"{H3PX_DIR}/nclimgrid", f"{H3PX_DIR}/grace", f"{H3PX_DIR}/gridmet",
                                                  f"{H3PX_DIR}/hls")
for _d in (CSB_DIR, STAGE4_DIR, NORMALS_DIR, NCLIMGRID_DIR, GRACE_DIR, GRIDMET_DIR, HLS_DIR):
    dbutils.fs.mkdirs(_d)
print("H3PX_DIR =", H3PX_DIR)

In [ ]:
# Idempotent managed-Delta materializer (Serverless-safe cache() stand-in). BIGINT *_cellid columns get a STRING
# twin for map tools: attr_h3_cellid -> attr_h3, field_h3_cellid -> field_h3, cellid -> h3.
from pyspark.sql.types import LongType


def _twin_name(c):
    return c[: -len("_cellid")] if c.endswith("_h3_cellid") else c.replace("cellid", "h3")


def _persist(df, name, cluster_by=None, rewrite=False):
    cell_cols = [f.name for f in df.schema.fields if isinstance(f.dataType, LongType) and f.name.endswith("cellid")]
    cols = []
    for c in df.columns:
        cols.append(F.col(c))
        if c in cell_cols and _twin_name(c) not in df.columns:
            cols.append(DBF.h3_h3tostring(F.col(c)).alias(_twin_name(c)))
    df = df.select(cols)
    if not (FORCE_REBUILD or rewrite) and spark.catalog.tableExists(name):
        if spark.table(name).columns == df.columns:
            print(f"... table {name} exists (skip; FORCE_REBUILD=False)")
            return spark.table(name)
        print(f"... {name}: columns changed since it was written; rewriting it")
    writer = df.write.mode("overwrite").option("overwriteSchema", "true")
    if cluster_by:
        writer = writer.clusterBy(*cluster_by)
    writer.saveAsTable(name)
    print(f"... wrote table {name} ({spark.table(name).count():,} rows)")
    return spark.table(name)

In [ ]:
# Downloaders and the shared raster helpers.
from databricks.labs.gbx.sample import (CsbDownloader, GraceDaDownloader, GridmetDownloader, HlsDownloader,
                                        NclimgridDownloader, StageIvDownloader)
from databricks.labs.gbx.sample import stage4 as stage4_source

stage4 = StageIvDownloader(template=STAGE4_URL_TEMPLATE)
gridmet = GridmetDownloader(base=GRIDMET_BASE)
nclim = NclimgridDownloader(normals_url=NORMALS_URL, monthly_url=NCLIMGRID_URL)
grace, hls, csb = GraceDaDownloader(), HlsDownloader(), CsbDownloader(href=CSB_ZIP_URL)
STAGE4_TO_MM = stage4_source.TO_MM  # mm per unit of the Stage IV daily total (the file's units attribute)

# A Stage IV 24 h total ends 12Z on its file date and counts for the day before.
STAGE4_DATE_SQL = "date_sub(to_date(regexp_extract(source, 'nws_precip_1day_(\\\\d{8})', 1), 'yyyyMMdd'), 1)"

# NDVI with Fmask bit 1 (cloud) masked, in GDAL's muparser (no bitwise operators and no floor: bit 1 of C is
# rint(C / 2 - 0.5) - 2 * rint(C / 4 - 0.5)). Fmask 255 (fill) has bit 1 set, so it is masked too. A zero red + NIR
# sum and a red or NIR NoData (-9999) give NoData too: no division by zero, no NDVI of a fill value.
NDVI_EXPR = ("((rint(C / 2 - 0.5) - 2 * rint(C / 4 - 0.5)) == 1 || (A + B) == 0 || A == -9999 || B == -9999)"
             " ? -9999 : (B - A) / (B + A)")
assert tuple(HLS_MASK_BITS) == (1,) or NDVI_ENGINE == "derivedband", \
    "NDVI_EXPR masks Fmask bit 1 only: set NDVI_ENGINE = 'derivedband' to mask other bits"
NDVI_PYFUNC = f'''
def ndvi_masked(in_ar, out_ar, xoff, yoff, xsize, ysize, raster_xsize, raster_ysize, buf_radius, gt, **kwargs):
    import numpy as np
    red, nir, fm = (a.astype("float64") for a in in_ar)
    bad = (fm == 255) | (red == -9999) | (nir == -9999) | ((nir + red) == 0)
    for b in {tuple(HLS_MASK_BITS)}:
        bad |= ((fm.astype("int64") >> b) & 1) == 1
    out_ar[:] = np.where(bad, np.nan, (nir - red) / np.where((nir + red) == 0, 1.0, nir + red))
'''


def ndvi_sql(red, nir, fmask):
    """SQL for the NDVI tile of three tile expressions (red, NIR, Fmask), the Fmask bits in HLS_MASK_BITS masked."""
    if NDVI_ENGINE == "mapalgebra":
        return f"gbx_rst_mapalgebra(array({red}, {nir}, {fmask}), '{NDVI_EXPR}')"
    src = NDVI_PYFUNC.replace("'", "''")
    return f"gbx_rst_derivedband(gbx_rst_frombands(array({red}, {nir}, {fmask})), '{src}', 'ndvi_masked')"


def centre_pixel(src):
    """(area m2, lon, lat) of the centre pixel of a tile's bytes or a raster path. The area is the library's own
    measure (h3fit.pixel_area_m2, the geodesic area on WGS84 that gbx_rst_h3_fitresolution uses)."""
    import rasterio
    from rasterio.io import MemoryFile
    from rasterio.warp import transform as warp_points

    from databricks.labs.gbx.pyrx.core.h3fit import pixel_area_m2

    def measure(ds):
        x, y = ds.transform * (ds.width // 2 + 0.5, ds.height // 2 + 0.5)
        if ds.crs is not None:
            (x,), (y,) = warp_points(ds.crs, "EPSG:4326", [x], [y])
        return pixel_area_m2(ds), x, y

    if isinstance(src, str):
        with rasterio.open(src) as ds:
            return measure(ds)
    with MemoryFile(bytes(src)) as mf, mf.open() as ds:
        return measure(ds)

In [ ]:
# Dataset tiles clipped to the AOI: one row per (dataset, collect_date) with the rule that turns a band into a date.
def _clip(df):
    return df.withColumn("tile", F.expr(f"gbx_rst_clip(tile, '{aoi_wkt()}', true, 'EPSG:4326')"))


def dataset_tiles(dataset):
    bbox = get_aoi_bbox()
    if dataset == "stage4":
        df = stage4.read(spark, STAGE4_DIR).withColumn("collect_date", F.expr(STAGE4_DATE_SQL))
        df = df.where(F.year("collect_date").isin(SEASONS)).withColumn("band_date_rule", F.lit("tile_date"))
        return _clip(df).select(F.lit("stage4").alias("dataset"), "collect_date", "band_date_rule", "tile")
    if dataset == "gridmet":
        frames = [gridmet.read(spark, GRIDMET_DIR, bbox, s).withColumn("collect_date", F.make_date(F.lit(s), F.lit(1), F.lit(1)))
                  for s in SEASONS]
        df = functools.reduce(DataFrame.unionByName, frames).withColumn("band_date_rule", F.lit("day_of_year"))
        return df.select(F.lit("gridmet").alias("dataset"), "collect_date", "band_date_rule", "tile")
    if dataset == "normals":
        frames = [nclim.read_normals(spark, NORMALS_DIR, bbox).withColumn("collect_date", F.make_date(F.lit(s), F.lit(1), F.lit(1)))
                  for s in SEASONS]  # the normals are dated in each season, so they join like the season's data
        df = functools.reduce(DataFrame.unionByName, frames).withColumn("band_date_rule", F.lit("month_offset"))
        return df.select(F.lit("normals").alias("dataset"), "collect_date", "band_date_rule", "tile")
    if dataset == "nclimgrid":  # the monthly cross-check (USE_NCLIMGRID): one band per month from January 1895
        df = nclim.read_monthly(spark, NCLIMGRID_DIR, bbox).withColumn("collect_date", F.make_date(F.lit(1895), F.lit(1), F.lit(1)))
        df = df.withColumn("band_date_rule", F.lit("month_offset"))
        return df.select(F.lit("nclimgrid").alias("dataset"), "collect_date", "band_date_rule", "tile")
    if dataset in ("gws_inst", "sfsm_inst"):
        df = grace.read(spark, GRACE_DIR, dataset, bbox).withColumn(
            "collect_date", F.to_date(F.regexp_extract("source", r"\.A(\d{8})\.", 1), "yyyyMMdd"))
        df = df.where(F.year("collect_date").isin(SEASONS)).withColumn("band_date_rule", F.lit("tile_date"))
        return df.select(F.lit(dataset).alias("dataset"), "collect_date", "band_date_rule", "tile")
    if dataset in ("hls_l30", "hls_s30"):
        sc = hls.read_scenes(spark, HLS_DIR).where(F.col("sensor") == dataset[-3:].upper())
        sc = sc.where(F.year("collect_date").isin(SEASONS)).repartition(FANOUT, "scene_id")
        w = aoi_wkt()
        tiles = [f"gbx_rst_fromfile({c}, 'GTiff', '{w}', 'EPSG:4326')" for c in ("red_path", "nir_path", "fmask_path")]
        df = sc.selectExpr("collect_date", "scene_id", f"{ndvi_sql(*tiles)} AS tile")
        return df.select(F.lit(dataset).alias("dataset"), "collect_date", F.lit("tile_date").alias("band_date_rule"), "tile")
    raise ValueError(f"unknown dataset {dataset!r}")


def band_to_date_sql(rule, band_col="band"):
    """SQL for the date of band `band_col` of a tile dated `collect_date`, by its rule."""
    return {"tile_date": "collect_date",
            "day_of_year": f"date_add(collect_date, {band_col} - 1)",
            "month_offset": f"add_months(collect_date, {band_col} - 1)"}[rule]


# Dataset -> scored attribute, in one place for NB02, NB03 and NB06:
# (dataset_tiles key, dataset label, attr_name, monthly reduction, months, scale to the attribute's units).
ATTRIBUTES = [
    ("stage4", "stage4", "precip_mm", "sum", MONTHS["stage4"], STAGE4_TO_MM),
    ("normals", "normals", "precip_normal_mm", "avg", MONTHS["normals"], NORMALS_TO_MM),
    ("gridmet", "gridmet", "wind_speed_ms", "avg", MONTHS["gridmet"], WIND_SCALE),
]
if USE_NCLIMGRID:
    ATTRIBUTES.append(("nclimgrid", "nclimgrid", "precip_mm", "avg", MONTHS["nclimgrid"], 1.0))
if EARTHDATA_READY:
    ATTRIBUTES += [("gws_inst", "grace", "gws_inst", "avg", MONTHS["gws_inst"], 1.0),
                   ("sfsm_inst", "grace", "sfsm_inst", "avg", MONTHS["sfsm_inst"], 1.0)]


def monthly_reduce(df, value_col, how, months, scale, keys, extra=()):
    """Daily or weekly values (a `day` column) to months: `how` ('sum' or 'avg') of `value_col` times `scale`, per
    `keys` and month, for `months` of SEASONS. Returns keys, collect_date (first of the month), measure, n_days
    (the distinct days or weeks with a value in the month) and extra."""
    agg = {"sum": F.sum, "avg": F.avg}[how]
    days = F.countDistinct(F.when(F.col(value_col).isNotNull(), F.col("day"))).alias("n_days")
    return (df.where(F.month("day").isin(list(months)) & F.year("day").isin(SEASONS))
            .groupBy(*keys, F.trunc("day", "month").alias("collect_date"))
            .agg((agg(value_col) * F.lit(scale)).alias("measure"), days, *extra))

In [ ]:
# Static-vs-interactive view helpers, gated by INTERACTIVE_PLOTS. The numbered notebooks render through these:
#   show_cells - an H3 cell choropleth: static (cells_as_gdf + an Esri basemap) or interactive MapLibre (grid_layer);
#                vmin / vmax fix the colour range of the static map (e.g. -m and m centre a diverging cmap on 0)
#   show_tile  - a raster tile over a basemap (plot_tile); static in both modes
from databricks.labs.gbx.vizx import cells_as_gdf, grid_layer, plot_interactive, plot_tile


def show_cells(df, cell_col, column, title=None, cmap="viridis", center=None, zoom=11, vmin=None, vmax=None):
    if INTERACTIVE_PLOTS:
        return plot_interactive(
            [grid_layer(df, grid_system="h3", cellid_col=cell_col, column=column, cmap=cmap, opacity=0.7,
                        label=title or column)],
            center=center, zoom=zoom,
        )
    import contextily as cx

    gdf = cells_as_gdf(df, cell_col=cell_col, extra_cols=[column], max_rows=None).to_crs(3857)
    ax = gdf.plot(column=column, legend=True, figsize=(9, 9), cmap=cmap, vmin=vmin, vmax=vmax, alpha=0.7,
                  edgecolor="black", linewidth=0.1)
    minx, miny, maxx, maxy = gdf.total_bounds
    pad = 0.2 * max(maxx - minx, maxy - miny)  # a little context around the cells
    ax.set_xlim(minx - pad, maxx + pad)
    ax.set_ylim(miny - pad, maxy + pad)
    cx.add_basemap(ax, source=cx.providers.Esri.WorldGrayCanvas)
    ax.set_axis_off()
    if title:
        ax.set_title(title)
    return ax


def show_tile(tile, **kw):
    return plot_tile(tile, **kw)


print("... view helpers: show_cells, show_tile (gated by INTERACTIVE_PLOTS)")